In [0]:
file_path = "/Workspace/sales ELT/sales_data.csv"

df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(file_path)

display(df)


Order_ID,Order_Date,Region,State,City,Category,Sub_Category,Product_Name,Quantity,Unit_Price,Discount,Profit,Sales
CA-2025-0001,2025-01-03,West,California,Los Angeles,Furniture,Chairs,Ergonomic Chair,2,180.0,0.1,45.0,324.0
CA-2025-0002,2025-01-04,East,New York,New York City,Technology,Phones,Wireless Headset,3,89.99,0.0,35.0,269.97
CA-2025-0003,2025-01-05,Central,Texas,Dallas,Office Supplies,Binders,Ring Binder Pack,5,12.5,0.05,8.0,59.38
CA-2025-0004,2025-01-06,South,Florida,Miami,Furniture,Tables,Compact Work Table,1,249.0,0.15,-12.0,211.65
CA-2025-0005,2025-01-08,West,Washington,Seattle,Technology,Accessories,USB-C Hub,4,39.99,0.0,42.0,159.96
CA-2025-0006,2025-01-10,East,Pennsylvania,null,Office Supplies,Paper,Copy Paper Box,8,24.0,0.0,28.0,192.0
CA-2025-0007,2025-01-12,Central,Illinois,Chicago,Technology,Phones,Desk Phone,2,75.0,0.1,18.0,135.0
CA-2025-0008,2025-01-14,South,Georgia,Atlanta,Furniture,Bookcases,Small Bookcase,1,135.0,0.05,9.0,128.25
CA-2025-0009,2025-01-16,West,Oregon,Portland,Office Supplies,Storage,Storage Bins,6,18.75,0.0,24.0,112.5
CA-2025-0010,2025-01-18,East,New Jersey,Newark,Technology,Accessories,Wireless Mouse,5,22.99,0.05,31.0,109.2


Bronze Layer

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("bronze_sales")

print("Bronze layer created")

Bronze layer created


Silver Layer

In [0]:
from pyspark.sql import functions as F

bronze = spark.table("bronze_sales")

silver = (
    bronze
    .dropDuplicates()
    .withColumn("Product_Name", F.trim(F.col("Product_Name")))
    .withColumn("City", F.coalesce(F.col("City"), F.lit("Unknown")))
    .withColumn("Order_Date", F.to_date("Order_Date"))
    .withColumn("Sales", F.col("Sales").cast("double"))
    .withColumn("Profit", F.col("Profit").cast("double"))
    .filter(
        F.col("Order_ID").isNotNull() &
        F.col("Sales").isNotNull() &
        (F.col("Sales") >= 0)
    )
    .withColumn(
        "Profit_Margin",
        F.round(F.col("Profit") / F.col("Sales") * 100, 2)
    )
)

silver.write.format("delta").mode("overwrite").saveAsTable("silver_sales")

print("Silver layer created. Rows:", silver.count())

Silver layer created. Rows: 30


Gold Layer

In [0]:
silver = spark.table("silver_sales")

gold = silver.groupBy("Region").agg(
    F.round(F.sum("Sales"), 2).alias("Total_Sales"),
    F.round(F.sum("Profit"), 2).alias("Total_Profit"),
    F.countDistinct("Order_ID").alias("Order_Count")
)

gold.write.format("delta").mode("overwrite").saveAsTable("gold_sales_by_region")

display(gold.orderBy(F.desc("Total_Sales")))

Region,Total_Sales,Total_Profit,Order_Count
West,1606.71,277.0,8
East,1449.27,139.0,8
South,1428.45,121.0,7
Central,899.68,158.0,7


In [0]:
%sql
SELECT
    Category,
    ROUND(SUM(Sales), 2) AS Total_Sales,
    ROUND(SUM(Profit), 2) AS Total_Profit
FROM silver_sales
GROUP BY Category
ORDER BY Total_Sales DESC;

In [0]:
for table in ["bronze_sales", "silver_sales", "gold_sales_by_region"]:
    print(f"{table}: {spark.table(table).count()} rows")

bronze_sales: 32 rows
silver_sales: 30 rows
gold_sales_by_region: 4 rows
